# 1. Imports and File selection 

In [ ]:
import io
import ipywidgets as widgets
import math
import numpy
import psycopg
import pandas as pd
import requests
import sqlite3
import sys
import tqdm
import warnings

from statsmodels.stats.multitest import fdrcorrection as fdr
from backend_config import load_config
from ipyfilechooser import FileChooser
from scipy import stats
from scipy.stats import ttest_ind
from sqlalchemy import create_engine
from sqlalchemy.dialects.postgresql import insert
from sqlite3 import Error
from sqlite3 import IntegrityError

## Select Baseline .csv File

In [ ]:
starting_directory = '/Volumes'
baseline_chooser = FileChooser(starting_directory)
display(baseline_chooser)

## Select Tap .csv File

In [ ]:
tap_chooser=FileChooser('/Volumes')
display(tap_chooser)

## Select Post Stimulus Arousal .csv File

In [ ]:
psa_chooser = FileChooser('/Volumes')
display(psa_chooser)

In [ ]:
screens = ['PD_Screen', 'ASD_Screen', 'G-Proteins_Screen', 'Glia_Genes_Screen', 
           'Neuron_Genes_Screen', 'PD_GWAS_Locus22_Screen', 'PD_GWAS_Locus71_Screen', 'ASD_WGS_Screen', 'Miscellaneous']

screen_chooser = widgets.Select(options=screens, value=screens[0], description='Screen:')
display(screen_chooser)

In [ ]:
Screen=screen_chooser.value
folder_path=baseline_chooser.selected_path
print(folder_path)

## Read baseline, tap and post stimulus arousal (psa) data

In [ ]:
# Read the baseline file
baseline_output = pd.read_csv(baseline_chooser.selected, index_col=0)#.drop(columns=['index'])

print(f"\nShape of the baseline .csv file: {baseline_output.shape}")

# Print the first five rows of the file
baseline_output.head()

In [ ]:
# Read the tap file
tap_output = pd.read_csv(tap_chooser.selected, index_col=0)

print(f"\nShape of the psa .csv file: {tap_output.shape}")

# Print the first five rows of the file
tap_output.head()

In [ ]:
# Read the psa file
psa_output = pd.read_csv(psa_chooser.selected, index_col=0)

for cols in ['Speed',
            #   'Interval Speed',
       'Bias', 'Morphwidth', 'Midline', 'Area', 'Angular Speed',
       'Aspect Ratio', 'Kink', 'Curve', 'Crab', 'Pathlength']:
    psa_output.rename(columns={cols: f"PSA {cols}"}, inplace=True)

print(f"\nShape of the tap .csv file: {psa_output.shape}")

# Print the first five rows of the file
psa_output.head()

In [ ]:
psa_output.columns

### Merge PSA with Tap response

In [ ]:
tap_psa_output = pd.merge(
    tap_output, psa_output.drop(columns=['Experiment', 'Time', 'Tap', 'PSA Morphwidth', 
                                         'PSA Midline', 'PSA Area', 'PSA Angular Speed',]),
    how='outer', 
    on=['dataset', 'Gene', 'Allele', 'Date', 'Plate_id', 'Screen', "taps" ] 
)

tap_psa_output = tap_psa_output[['dataset', 'Gene', 'Allele', 'Date', 'Plate_id', 'plate', 
                                 'Screen', 'taps', 'time', 'dura', 'dist', 'prob', 'speed',
                                 'PSA Speed', 
                                #  'PSA Interval Speed', 
                                 'PSA Bias',
                                 'PSA Aspect Ratio', 'PSA Kink', 'PSA Curve', 'PSA Crab'
                                 ]]

print(f"Shape of the dataframe: {tap_psa_output.shape}")

tap_psa_output.rename(columns={
    # 'prob': 'Probability',
    # 'dura': 'Duration',
    # 'speed': 'Speed'
}, inplace=True)

tap_psa_output.head()

In [ ]:
# tap_psa_output.to_csv("tap_psa_output.csv")

# 2. DataFrame preparation

### 2.1. Tap & PSA Data

In [ ]:
# Dataframe for first tap
PD_first_tap = (
    tap_psa_output[(tap_psa_output.taps==1)]
    .reset_index().drop(columns="index")
    .rename(columns={"dura": "init_dura", "prob": "init_prob", "speed": "init_speed",
                     "PSA Speed": "init_psa_speed",
                     "PSA Bias": "init_psa_bias",
                     "PSA Aspect Ratio": "init_psa_aspect_ratio",
                     "PSA Kink": "init_psa_kink",
                     "PSA Curve": "init_psa_curve",
                     "PSA Crab": "init_psa_crab"}))
PD_first_tap.head()

In [ ]:
# Dataframe for recovery taps
PD_recov_taps = (
    tap_psa_output[(tap_psa_output.taps==31)]
    .reset_index().drop(columns="index")
    .rename(columns={"dura": "recov_dura", "prob": "recov_prob", "speed":"recov_speed",
                     "PSA Speed": "recov_psa_speed",
                     "PSA Bias": "recov_psa_bias",
                     "PSA Aspect Ratio": "recov_psa_aspect_ratio",
                     "PSA Kink": "recov_psa_kink",
                     "PSA Curve": "recov_psa_curve",
                     "PSA Crab": "recov_psa_crab"})
)

PD_recov_taps.head()

In [ ]:
# Dataframe for last three taps
PD_final_taps = (
    tap_psa_output[(tap_psa_output.taps >= 28) & (tap_psa_output.taps <= 30)]
    .groupby(["dataset", "Date","Plate_id","Screen","Gene","Allele","plate"])
    .mean()
    .reset_index()
    .rename(columns={"dura": "final_dura", "prob": "final_prob", "speed": "final_speed",
                     "PSA Speed": "final_psa_speed",
                     "PSA Bias": "final_psa_bias",
                     "PSA Aspect Ratio": "final_psa_aspect_ratio",
                     "PSA Kink": "final_psa_kink",
                     "PSA Curve": "final_psa_curve",
                     "PSA Crab": "final_psa_crab"}, errors="raise")
)

PD_final_taps.head()

In [ ]:
# Dataframe to analyse habituation behaviour after merging first tap and final taps

PD_habit_levels = pd.merge(
    PD_first_tap, 
    PD_final_taps, 
    on =['dataset', 'plate', "Plate_id", "Screen", "Gene", "Allele", "Date"], how ='left'
).drop(columns=['time_x','time_y','dist_x','dist_y', 'taps_x', 'taps_y']).dropna()

PD_habit_levels['habit_dura'] = PD_habit_levels['init_dura'] - PD_habit_levels['final_dura']
PD_habit_levels['habit_prob'] = PD_habit_levels['init_prob'] - PD_habit_levels['final_prob']
PD_habit_levels['habit_speed'] = PD_habit_levels['init_speed'] - PD_habit_levels['final_speed']
PD_habit_levels['habit_psa_speed'] = PD_habit_levels['init_psa_speed'] - PD_habit_levels['final_psa_speed']
PD_habit_levels['habit_psa_bias'] = PD_habit_levels['init_psa_bias'] - PD_habit_levels['final_psa_bias']
PD_habit_levels['habit_psa_aspect_ratio'] = PD_habit_levels['init_psa_aspect_ratio'] - PD_habit_levels['final_psa_aspect_ratio']
PD_habit_levels['habit_psa_kink'] = PD_habit_levels['init_psa_kink'] - PD_habit_levels['final_psa_kink']
PD_habit_levels['habit_psa_curve'] = PD_habit_levels['init_psa_curve'] - PD_habit_levels['final_psa_curve']
PD_habit_levels['habit_psa_crab'] = PD_habit_levels['init_psa_crab'] - PD_habit_levels['final_psa_crab']

In [ ]:
# Continue to analyse habituation behaviour after merging with recovery taps

if PD_recov_taps.empty:
    PD_habituation = pd.merge(PD_habit_levels, PD_recov_taps, on =['dataset','plate',"Plate_id","Screen","Gene","Allele","Date"], how ='outer')
else:
    PD_habituation = pd.merge(PD_habit_levels, PD_recov_taps, on =['dataset','plate',"Plate_id","Screen","Gene","Allele","Date"], how ='left')

if Screen not in ['Neuron_Genes_Screen', 'G-Proteins_Screen']:
    PD_habituation = PD_habituation.dropna() 

PD_habituation['recovery_dura']=(PD_habituation.recov_dura-PD_habituation.init_dura)/PD_habituation.init_dura*100
PD_habituation['recovery_prob']=(PD_habituation.recov_prob-PD_habituation.init_prob)/PD_habituation.init_prob*100
PD_habituation['recovery_speed']=(PD_habituation.recov_speed-PD_habituation.init_speed)/PD_habituation.init_speed*100
PD_habituation['recovery_psa_speed']=(PD_habituation.recov_psa_speed-PD_habituation.init_psa_speed)/PD_habituation.init_psa_speed*100
PD_habituation['recovery_psa_bias']=(PD_habituation.recov_psa_bias-PD_habituation.init_psa_bias)/PD_habituation.init_psa_bias*100
PD_habituation['recovery_psa_aspect_ratio']=(PD_habituation.recov_psa_aspect_ratio-PD_habituation.init_psa_aspect_ratio)/PD_habituation.init_psa_aspect_ratio*100
PD_habituation['recovery_psa_kink']=(PD_habituation.recov_psa_kink-PD_habituation.init_psa_kink)/PD_habituation.init_psa_kink*100
PD_habituation['recovery_psa_curve']=(PD_habituation.recov_psa_curve-PD_habituation.init_psa_curve)/PD_habituation.init_psa_curve*100
PD_habituation['recovery_psa_crab']=(PD_habituation.recov_psa_crab-PD_habituation.init_psa_crab)/PD_habituation.init_psa_crab*100

PD_habituation['memory_retention_dura']=(PD_habituation.recov_dura-PD_habituation.final_dura)
PD_habituation['memory_retention_prob']=(PD_habituation.recov_prob-PD_habituation.final_prob)
PD_habituation['memory_retention_speed']=(PD_habituation.recov_speed-PD_habituation.final_speed)
PD_habituation['memory_retention_psa_speed']=(PD_habituation.recov_psa_speed-PD_habituation.final_psa_speed)
PD_habituation['memory_retention_psa_bias']=(PD_habituation.recov_psa_bias-PD_habituation.final_psa_bias)
PD_habituation['memory_retention_psa_aspect_ratio']=(PD_habituation.recov_psa_aspect_ratio-PD_habituation.final_psa_aspect_ratio)
PD_habituation['memory_retention_psa_kink']=(PD_habituation.recov_psa_kink-PD_habituation.final_psa_kink)
PD_habituation['memory_retention_psa_curve']=(PD_habituation.recov_psa_curve-PD_habituation.final_psa_curve)
PD_habituation['memory_retention_psa_crab']=(PD_habituation.recov_psa_crab-PD_habituation.final_psa_crab)


# Rename `PD_habituation` to `tap_data` based on the condition below
if Screen in ['Neuron_Genes_Screen', 'G-Proteins_Screen']:
    tap_data=PD_habituation.dropna(subset = ['dataset', 'Gene', 'Allele','Date', 'Plate_id', 'plate', 'Screen', 
                                             'init_dura', 'init_prob', 'init_speed', 'init_psa_speed', 'init_psa_bias',
                                             'init_psa_aspect_ratio','init_psa_kink','init_psa_curve','init_psa_crab',
                                            'final_dura', 'final_prob', 'final_psa_speed', 'final_psa_bias',
                                            'final_psa_aspect_ratio','final_psa_kink','final_psa_curve','final_psa_crab',
                                            'final_speed', 'habit_dura', 'habit_prob', 'habit_speed','habit_psa_speed','habit_psa_bias',
                                            'habit_psa_aspect_ratio','habit_psa_kink','habit_psa_curve','habit_psa_crab'])
else:
    tap_data=PD_habituation.dropna() 


# Display final dataframe
tap_data.head()


In [ ]:
tap_psa_output

In [ ]:
tap_psa_output_max = tap_psa_output.groupby(['dataset', 'Gene', 'Allele', 'Date', 'Plate_id', 'plate', 'Screen'], as_index = False).max().round(4).rename(
    columns = {
        'PSA Speed': 'max_psa_speed',
        'PSA Bias': 'max_psa_bias',
        'PSA Aspect Ratio': 'max_psa_aspect_ratio',
        'PSA Kink': 'max_psa_kink',
        'PSA Curve': 'max_psa_curve',
        'PSA Crab': 'max_psa_crab'
    }).drop(columns = ['taps','time','dura','dist','prob','speed'])

tap_psa_output_mean = tap_psa_output.groupby(['dataset','Gene','Allele','Date','Plate_id','plate','Screen'], as_index = False).mean().round(4).rename(
    columns = {
        'prob': 'mean_prob',
        'dura': 'mean_dura',
        'speed': 'mean_speed',
        'PSA Speed': 'mean_psa_speed',
        'PSA Bias': 'mean_psa_bias',
        'PSA Aspect Ratio': 'mean_psa_aspect_ratio',
        'PSA Kink': 'mean_psa_kink',
        'PSA Curve': 'mean_psa_curve',
        'PSA Crab': 'mean_psa_crab'
    }).drop(columns = ['taps','time','dist'])

tap_psa_output_mean.head()

In [ ]:
tap_data = pd.merge(PD_habituation, tap_psa_output_max, on =['dataset','plate',"Plate_id","Screen","Gene","Allele","Date"], how ='outer')
tap_data = pd.merge(tap_data, tap_psa_output_mean, on =['dataset','plate',"Plate_id","Screen","Gene","Allele","Date"], how ='outer')

tap_data['sensitization_psa_speed']=(tap_data.max_psa_speed - tap_data.init_psa_speed)
tap_data['sensitization_psa_bias']=(tap_data.max_psa_bias - tap_data.init_psa_bias)
tap_data['sensitization_psa_aspect_ratio']=(tap_data.max_psa_aspect_ratio - tap_data.init_psa_aspect_ratio)   
tap_data['sensitization_psa_kink']=(tap_data.max_psa_kink - tap_data.init_psa_kink)
tap_data['sensitization_psa_curve']=(tap_data.max_psa_curve - tap_data.init_psa_curve)
tap_data['sensitization_psa_crab']=(tap_data.max_psa_crab - tap_data.init_psa_crab)

tap_data

In [ ]:
print(tap_data.columns.to_list())

### 2.2. PSA data

In [ ]:
# # function to calculate Initial, Final, Peak, ect values for specified column (metric)

# def summary_metrics(df, metric = 'PSA Speed'):

#     initial = df[metric].iloc[0]
#     # peak = df[metric].max()
#     peak_id = df[metric].values.argmax() # only used for peak tap calculation
#     peak_tap = df['taps'].iloc[peak_id]
#     # mean = df[metric].mean()
#     sensitization = peak - initial
#     # recovery = df[df['taps']==31][metric] if not df[df['taps']==31].empty else 0
#     # final = df[((df.taps >= 28) & (df.taps <= 30))][metric]
#     # spontaneous_recovery = 100*(initial - recovery)/initial if metric not in ['PSA Aspect Ratio', 'PSA Kink', 'PSA Curve', 'PSA Crab'] else 100*(recovery - initial)/initial
#     # memory_retention = final - recovery if metric not in ['PSA Aspect Ratio', 'PSA Kink', 'PSA Curve', 'PSA Crab'] else recovery - final

    
#     # initial_to_peak = df[metric].iloc[: peak_id+1].mean()
#     # peak_to_recovery = df[metric].iloc[peak_id:].mean()

#     # print(df)
#     # print("-----------------")

    

#     return pd.Series({
#         f'Initial {metric}': initial, 
#         # f'Final {metric}': final,
#         # f'Recovery {metric}': recovery, 
#         f'Peak {metric}': peak,
#         f'Peak Tap Number of {metric}': peak_tap,
#         f'Average {metric}': mean,
#         f'Sensitization of {metric}': sensitization,
#         # f'Habituation of {metric}': habituation,
#         # f'Spontaneous Recovery of {metric}': spontaneous_recovery,
#         # f'Memory Retention of {metric}': memory_retention
#         # f'Initial_to_peak {metric}': initial_to_peak, 
#         # f'Peak_to_recovery {metric}': peak_to_recovery
#         })

In [ ]:
# warnings.filterwarnings('ignore')

# # columns to summarize
# metrics_to_summarize = ['PSA Speed', 'PSA Bias', 'PSA Angular Speed', 
#                         'PSA Aspect Ratio', 'PSA Kink', 'PSA Curve', 'PSA Crab']

# # standard columns
# group_cols = ['Experiment', 'Plate_id', 'Date', 'Screen', 'dataset', 'Gene', 'Allele', 'taps']

# # pass each column to summarise through `summary_metrics` function and merge the summarised values to psa_output
# psa_data = psa_output[group_cols].drop_duplicates()
# for metric in metrics_to_summarize:
#     summary = psa_output.groupby(group_cols).apply(lambda x: summary_metrics(x, metric)).reset_index()
#     psa_data = pd.merge(psa_data, summary, on=group_cols, how='left')

In [ ]:
# psa_data.head()

# 3. Run Statistics (T-Test and mean sample distance) on Data

## 3.1 Generate dataframes conditioned by `baseline` (True/False) and `allele` (True/False)

In [ ]:
def get_output_byplate(output, baseline=["true", "false", "psa"], allele = [False, True]):
    """
    Aggregates data by 'Gene' or 'Allele' and drops 'Plate_id','Date','Screen','dataset', etc

    Parameters:
        output (pd.DataFrame): Input DataFrame (either baseline_output or tap_data)
        baseline (boolean): whether data is baseline (True) or tap response (False)
        allele (boolean): group by allele (True) or group by gene (False)

    Returns:
        A DataFrame with plate-level averages
    """
    
    # columns to delete if baseline = true (baseline)
    if baseline == "true":
        drop_col = ['Plate_id','n','Number','Time','Screen','Date','Allele']
    # columns to delete if baseline = false (tap and psa data)
    elif baseline == "false":
        drop_col = ['Plate_id','Screen','Date','Allele','dist','plate','time',
                       'taps','recov_dura','recov_prob','recov_speed']
    # columns to delete if baseline = psa (psa)
    else: 
        # drop_col = ['Experiment', 'Plate_id', 'Date', 'Screen', 'Allele']
        pass

    drop_col.append('Gene') if allele else drop_col.append('dataset')
     
    output_byplate = output.groupby(
        by=['Plate_id','Date','Screen','dataset','Gene','Allele'],
        as_index=False).mean().drop(columns=drop_col)
    
    return output_byplate

#### 3.1.1 `baseline` = True, `allele` = False

In [ ]:
baseline_output_byplate=get_output_byplate(baseline_output, baseline= "true", allele=False)

print(f"Shape: {baseline_output_byplate.shape}")

baseline_output_byplate.head()

#### 3.1.2 `baseline` = False, `allele` = False

In [ ]:
tap_data_byplate=get_output_byplate(tap_data, baseline="false", allele=False)

print(f"Shape: {tap_data_byplate.shape}")

tap_data_byplate.head()

#### 3.1.3 `baseline` = True, `allele` = True

In [ ]:
baseline_output_allele_byplate = get_output_byplate(baseline_output,baseline="true", allele=True)

print(f"Shape: {baseline_output_allele_byplate.shape}")

baseline_output_allele_byplate.head()

#### 3.1.4 `baseline` = False, `allele` = True

In [ ]:
tap_data_allele_byplate = get_output_byplate(tap_data, baseline="false", allele=True)

print(f"Shape: {tap_data_allele_byplate.shape}")

tap_data_allele_byplate.head()

In [ ]:
# tap_data_allele_byplate[tap_data_allele_byplate.dataset=='N2_XJ1']

#### 3.1.5 `baseline` = "psa" , `allele` = False *Defunct*

In [ ]:
# psa_data_byplate = get_output_byplate(psa_data, baseline="psa", allele=False)

# print(f"Shape: {psa_data_byplate.shape}")

# psa_data_byplate.head()

#### 3.1.6 `baseline` = "psa" , `allele` = True

In [ ]:
# psa_data_allele_byplate = get_output_byplate(psa_data, baseline="psa", allele=True)

# print(f"Shape: {psa_data_allele_byplate.shape}")

# psa_data_allele_byplate.head()

## 3.2 Calculate Mean Distances and CIs

In [ ]:

def extract_phenotypes(df):
    ''' 
    Splits a multi-column DataFrame into a list of DataFrames, each containing one phenotype

    input: 
        df (pd.DataFrame): dataframe with multiple columns (1st column is the index, the other are phenotypes)

    returns:
        list_phenotypes_df: list with 2 columns - one for index and one for phenotype, 
            for how many phenotypes there are in the input
    '''
    list_phenotypes_df = []
    index = df.columns[0]
    for i in df.columns[1:]:
        list_phenotypes_df.append(df[[index, i]].copy())

    return list_phenotypes_df



def ci95(df):
    """
    input: df of 4 columns: index, mean, count, std

    returns: df of 6 columns: index, mean, count, std, ci95_hi, ci95_low

    """
    for metric in df.columns.levels[0]:
        if metric == 'Gene':
            pass
        else:
            ci95_hi = []
            ci95_lo = []
            for i in df[metric].index:
                m = df[metric]['mean'].loc[i]
                c = df[metric]['count'].loc[i]
                s = df[metric]['sem'].loc[i]
                ci95_hi.append(stats.t.interval(confidence=0.95, df=c-1, loc=m, scale=s)[1])
                ci95_lo.append(stats.t.interval(confidence=0.95, df=c-1, loc=m, scale=s)[0])
            df[metric,'ci95_hi'] = ci95_hi
            df[metric,'ci95_lo'] = ci95_lo
            # df[metric,'ci95']=list(zip(ci95_lo,ci95_hi))
            
    return df



def calculate_MSD(list_of_dfs, by):
    new_list_of_dfs = []
    
    for df in list_of_dfs:
        # Get phenotype column name (assuming 2nd column is the metric)
        pheno_col = df.columns[1]
        
        # Calculate statistics
        stats = df.groupby(by)[df.columns[1]].agg(['mean', 'count', 'sem'])

        
        # Convert to MultiIndex if needed (more robust version)
        if not isinstance(stats.columns, pd.MultiIndex):
            stats.columns = pd.MultiIndex.from_tuples([(pheno_col, col) for col in stats.columns])
        
        # Calculate CI
        stats_2 = ci95(stats)
        
        # Get N2 control data
        if Screen == "Neuron_Genes_Screen":
            N2_mask = stats_2.index == 'N2' if by == "Gene" else stats_2.index.isin(['N2_XJ1','N2_N2'])
        else:
            N2_mask = stats_2.index == 'N2'
            
        N2_data = stats_2[N2_mask]
        
        # Subtract N2 values
        stats_2.iloc[:, 0] -= N2_data.iloc[0, 0]  # mean
        stats_2.iloc[:, 3] -= N2_data.iloc[0, 0]  # ci95_hi
        stats_2.iloc[:, 4] -= N2_data.iloc[0, 0]  # ci95_low
        
        new_list_of_dfs.append(stats_2)
    
    return new_list_of_dfs

In [ ]:
def calculate_MSD(list_of_dfs, by):
    new_list_of_dfs = []
    
    for df in list_of_dfs:
        # Get phenotype column name (assuming 2nd column is the metric)
        pheno_col = df.columns[1]
        
        # Create proper MultiIndex structure
        stats = df.groupby(by)[df.columns[1]].agg(['mean', 'count', 'sem'])

        # Convert to MultiIndex if needed (more robust version)
        if not isinstance(stats.columns, pd.MultiIndex):
            stats.columns = pd.MultiIndex.from_tuples([(pheno_col, col) for col in stats.columns])
        
        # Calculate CIs
        stats_2 = ci95(stats)
        
        # Get N2 control data
        if Screen == "Neuron_Genes_Screen":
            N2_mask = stats_2.index == 'N2' if by == "Gene" else stats_2.index.isin(['N2_XJ1','N2_N2'])
        else:
            N2_mask = stats_2.index == 'N2'
            
        N2_data = stats_2[N2_mask]
        
        # Subtract N2 values
        stats_2.iloc[:, 0] -= N2_data.iloc[0, 0]  # mean
        stats_2.iloc[:, 3] -= N2_data.iloc[0, 0]  # ci95_hi
        stats_2.iloc[:, 4] -= N2_data.iloc[0, 0]  # ci95_low
        
        new_list_of_dfs.append(stats_2)
    
    return new_list_of_dfs

In [ ]:
def get_MSD(list_MSD):
    '''
    input: List of dataframes, each representing a phenotype with calculated MSD.

    returns: Single combined dataframe joining all input dataframes with MSD values.
    '''
    for a in list_MSD:
        if a.columns.levels[0] == list_MSD[0].columns.levels[0]:
            MSD=a
        else:
            MSD=MSD.join(a)
    return MSD

In [ ]:
def get_combined_MSD(baseline_byplate,tap_byplate, by=['Gene','dataset']):
    """
    Combines MSD datafram from baseline plates and tap plates

    input:
        - baseline_byplate: baseline data by plate
        - tap_byplate: tap data by plate
        - by: what to group by "Gene" or "dataset"
    returns:
        - combined MSD dataframe
    """
    list_baseline_MSD=calculate_MSD(extract_phenotypes(baseline_byplate), by=by)

    list_tap_MSD=calculate_MSD(extract_phenotypes(tap_byplate), by=by)

    # list_psa_MSD=calculate_MSD(extract_phenotypes(psa_byplate), by=by)

    baseline_MSD = get_MSD(list_baseline_MSD)
    
    tap_MSD = get_MSD(list_tap_MSD)
    # psa_MSD = get_MSD(list_psa_MSD)

    combined_MSD = pd.merge(baseline_MSD, tap_MSD, on=by, how='outer')

    combined_MSD=combined_MSD.rename(columns={"habit_dura":"Habituation of Response Duration",
                                         "habit_prob": "Habituation of Response Probability",
                                         "habit_speed":"Habituation of Response Speed",
                                         "habit_psa_speed":"Habituation of PSA Speed",
                                         "habit_psa_bias":"Habituation of PSA Bias",
                                         "habit_psa_aspect_ratio":"Habituation of PSA Aspect Ratio",
                                         "habit_psa_kink":"Habituation of PSA Kink",
                                         "habit_psa_curve":"Habituation of PSA Curve",
                                         "habit_psa_crab":"Habituation of PSA Crab",
                                         "init_dura": "Initial Response Duration",
                                         "init_prob": "Initial Response Probability",
                                         "init_speed": "Initial Response Speed",
                                         "init_psa_speed": "Initial PSA Speed",
                                         "init_psa_bias": "Initial PSA Bias",
                                         "init_psa_aspect_ratio": "Initial PSA Aspect Ratio",
                                         "init_psa_kink": "Initial PSA Kink",
                                         "init_psa_curve": "Initial PSA Curve",
                                         "init_psa_crab": "Initial PSA Crab",
                                         "final_dura": "Final Response Duration",
                                         "final_prob": "Final Response Probability",
                                         "final_speed": "Final Response Speed",
                                         "final_psa_speed": "Final PSA Speed",
                                         "final_psa_bias": "Final PSA Bias",
                                         "final_psa_aspect_ratio": "Final PSA Aspect Ratio",
                                         "final_psa_kink": "Final PSA Kink",
                                         "final_psa_curve": "Final PSA Curve",
                                         "final_psa_crab": "Final PSA Crab",
                                         "recovery_dura": "Spontaneous Recovery of Response Duration",
                                         "recovery_prob": "Spontaneous Recovery of Response Probability",
                                         "recovery_speed": "Spontaneous Recovery of Response Speed",
                                         "recovery_psa_speed": "Spontaneous Recovery of PSA Speed",
                                         "recovery_psa_bias": "Spontaneous Recovery of PSA Bias",
                                         "recovery_psa_aspect_ratio": "Spontaneous Recovery of PSA Aspect Ratio",
                                         "recovery_psa_kink": "Spontaneous Recovery of PSA Kink",
                                         "recovery_psa_curve": "Spontaneous Recovery of PSA Curve",
                                         "recovery_psa_crab": "Spontaneous Recovery of PSA Crab",
                                         "recov_psa_speed": "Spontaneous Recovery Stimulus of PSA Speed",
                                         "recov_psa_bias": "Spontaneous Recovery Stimulus of PSA Bias",
                                         "recov_psa_aspect_ratio": "Spontaneous Recovery Stimulus of PSA Aspect Ratio",
                                         "recov_psa_kink": "Spontaneous Recovery Stimulus of PSA Kink",
                                         "recov_psa_curve": "Spontaneous Recovery Stimulus of PSA Curve",
                                         "recov_psa_crab": "Spontaneous Recovery Stimulus of PSA Crab",
                                         "memory_retention_dura": "Memory Retention of Response Duration",
                                         "memory_retention_prob": "Memory Retention of Response Probability",
                                         "memory_retention_speed": "Memory Retention of Response Speed",
                                         "memory_retention_psa_speed": "Memory Retention of PSA Speed",
                                         "memory_retention_psa_bias": "Memory Retention of PSA Bias",
                                         "memory_retention_psa_aspect_ratio": "Memory Retention of PSA Aspect Ratio",
                                         "memory_retention_psa_kink": "Memory Retention of PSA Kink",
                                         "memory_retention_psa_curve": "Memory Retention of PSA Curve",
                                         "memory_retention_psa_crab": "Memory Retention of PSA Crab",
                                         "sensitization_psa_speed": "Sensitization of PSA Speed",
                                         "sensitization_psa_bias": "Sensitization of PSA Bias",
                                         "sensitization_psa_aspect_ratio": "Sensitization of PSA Aspect Ratio",
                                         "sensitization_psa_kink": "Sensitization of PSA Kink",
                                         "sensitization_psa_curve": "Sensitization of PSA Curve",
                                         "sensitization_psa_crab": "Sensitization of PSA Crab",
                                         "max_psa_speed": "Peak PSA Speed",
                                         "max_psa_bias": "Peak PSA Bias",
                                         "max_psa_aspect_ratio": "Peak PSA Aspect Ratio",
                                         "max_psa_kink": "Peak PSA Kink",
                                         "max_psa_curve": "Peak PSA Curve",
                                         "max_psa_crab": "Peak PSA Crab",
                                         "mean_psa_speed": "Average PSA Speed",
                                         "mean_psa_bias": "Average PSA Bias",
                                         "mean_psa_aspect_ratio": "Average PSA Aspect Ratio",
                                         "mean_psa_kink": "Average PSA Kink",
                                         "mean_psa_curve": "Average PSA Curve",
                                         "mean_psa_crab": "Average PSA Crab"})

    combined_MSD=combined_MSD.reset_index()
    combined_MSD.columns = combined_MSD.columns.to_flat_index().str.join('-')
    combined_MSD=combined_MSD.rename(columns={by+"-": by})
    combined_MSD['Screen']=Screen
    
    return combined_MSD

### 3.2.1 Gene-level MSD

In [ ]:
print(combined_MSD.columns.to_list())

In [ ]:
combined_MSD=get_combined_MSD(baseline_output_byplate,
                              tap_data_byplate, 
                            #   psa_data_byplate,
                              by='Gene')

combined_MSD.head()

### 3.2.2 Allele-level SMD

In [ ]:
allele_combined_MSD=get_combined_MSD(baseline_output_allele_byplate,
                                     tap_data_allele_byplate, 
                                    #  psa_data_allele_byplate,
                                     by='dataset')

allele_combined_MSD.head()

## 3.3 T-Stat analysis

In [ ]:
def baseline_metrics(by=["Gene","dataset"]):
    """
    Create a list of empty dataframe and list of metrics for baseline analysis

    input:
        by (list): what to group by "Gene" or "dataset"
        
    returns:
        list_baseline_Tstats: dataframes to store t-statistics
        list_baseline_metrics: dataframes to store metic names
    """
    PD_baseline_instantspeed_T=pd.DataFrame(columns = [by,"Speed", "Speed p-value"])
    # PD_baseline_intspeed_T=pd.DataFrame(columns = [by,"Interval Speed"])
    PD_baseline_bias_T=pd.DataFrame(columns = [by,"Bias", "Bias p-value"])
    PD_baseline_morphwidth_T=pd.DataFrame(columns = [by,"Morphwidth", "Morphwidth p-value"])
    PD_baseline_midline_T=pd.DataFrame(columns = [by,"Midline", "Midline p-value"])
    PD_baseline_area_T=pd.DataFrame(columns = [by,"Area", "Area p-value"])
    PD_baseline_angularspeed_T=pd.DataFrame(columns = [by,"Angular Speed", "Angular Speed p-value"])
    PD_baseline_aspectratio_T=pd.DataFrame(columns = [by,"Aspect Ratio", "Aspect Ratio p-value"])
    PD_baseline_kink_T=pd.DataFrame(columns = [by,"Kink", "Kink p-value"])
    PD_baseline_curve_T=pd.DataFrame(columns = [by,"Curve", "Curve p-value"])
    PD_baseline_crab_T=pd.DataFrame(columns = [by,"Crab", "Crab p-value"])
    PD_baseline_pathlength_T=pd.DataFrame(columns = [by,"Pathlength", "Pathlength p-value"])

    list_baseline_Tstats=[ 
                        PD_baseline_morphwidth_T,
                        PD_baseline_midline_T,
                        PD_baseline_area_T,
                        PD_baseline_aspectratio_T,
                        PD_baseline_instantspeed_T,
                        # PD_baseline_intspeed_T,
                        PD_baseline_angularspeed_T,
                        PD_baseline_bias_T,
                        PD_baseline_kink_T,
                        PD_baseline_curve_T,
                        PD_baseline_crab_T,
                        PD_baseline_pathlength_T]

    list_baseline_metrics=[
                        "Morphwidth",
                        "Midline",
                        "Area",
                        "Aspect Ratio",
                        "Speed",
                        # "Interval Speed",
                        "Angular Speed",
                        "Bias",
                        "Kink",
                        "Curve",
                        "Crab",
                        "Pathlength"]
    
    return list_baseline_Tstats, list_baseline_metrics

In [ ]:
def tap_metrics(by=["Gene","dataset"]):
    """
    Create a list of empty dataframes and list of metrics for tap analysis

    input:
        by (list): what to group by "Gene" or "dataset"
        
    returns:
        list_tap_Tstats: dataframes to store t-statistics
        list_tap_metrics: dataframes to store metic names
    """
    recovery_dura=pd.DataFrame(columns = [by,"Spontaneous Recovery of Response Duration", "Spontaneous Recovery of Response Duration p-value"])
    recovery_prob=pd.DataFrame(columns = [by,"Spontaneous Recovery of Response Probability", "Spontaneous Recovery of Response Probability p-value"])
    recovery_speed=pd.DataFrame(columns = [by,"Spontaneous Recovery of Response Speed", "Spontaneous Recovery of Response Speed p-value"])
    memory_retention_dura=pd.DataFrame(columns = [by,"Memory Retention of Response Duration", "Memory Retention of Response Duration p-value"])
    memory_retention_prob=pd.DataFrame(columns = [by,"Memory Retention of Response Probability", "Memory Retention of Response Probability p-value"])
    memory_retention_speed=pd.DataFrame(columns = [by,"Memory Retention of Response Speed", "Memory Retention of Response Speed p-value"])
    init_dura=pd.DataFrame(columns = [by,"Initial Response Duration", "Initial Response Duration p-value"])
    init_prob=pd.DataFrame(columns = [by,"Initial Response Probability", "Initial Response Probability p-value"])
    init_speed=pd.DataFrame(columns = [by,"Initial Response Speed", "Initial Response Speed p-value"])
    final_dura=pd.DataFrame(columns = [by,"Final Response Duration", "Final Response Duration p-value"])
    final_prob=pd.DataFrame(columns = [by,"Final Response Probability", "Final Response Probability p-value"])
    final_speed=pd.DataFrame(columns = [by,"Final Response Speed", "Final Response Speed p-value"])
    hab_dura=pd.DataFrame(columns = [by,"Habituation of Response Duration", "Habituation of Response Duration p-value"])
    hab_prob=pd.DataFrame(columns = [by,"Habituation of Response Probability", "Habituation of Response Probability p-value"])
    hab_speed=pd.DataFrame(columns = [by,"Habituation of Response Speed", "Habituation of Response Speed p-value"])

    psa_initial_speed = pd.DataFrame(columns=[by,"Initial PSA Speed", "Initial PSA Speed p-value"])
    psa_final_speed = pd.DataFrame(columns=[by,"Final PSA Speed", "Final PSA Speed p-value"])
    psa_recovery_speed = pd.DataFrame(columns=[by,"Recovery PSA Speed", "Recovery PSA Speed p-value"])
    psa_peak_speed = pd.DataFrame(columns=[by,"Peak PSA Speed", "Peak PSA Speed p-value"])
    # psa_peak_tap_speed = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Speed"])
    psa_avg_speed = pd.DataFrame(columns=[by,"Average PSA Speed", "Average PSA Speed p-value"])
    psa_sensitization_speed = pd.DataFrame(columns=[by,"Sensitization of PSA Speed", "Sensitization of PSA Speed p-value"])
    psa_habituation_speed = pd.DataFrame(columns=[by,"Habituation of PSA Speed", "Habituation of PSA Speed p-value"])
    psa_spontaneous_recovery_speed = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Speed", "Spontaneous Recovery of PSA Speed p-value"])
    psa_memory_retention_speed = pd.DataFrame(columns=[by,"Memory Retention of PSA Speed", "Memory Retention of PSA Speed p-value"])
    # psa_initial_to_peak_speed = pd.DataFrame(columns=[by,"Initial_to_peak PSA Speed"])
    # psa_peak_to_recovery_speed = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Speed"])

    psa_initial_bias = pd.DataFrame(columns=[by,"Initial PSA Bias", "Initial PSA Bias p-value"])
    psa_final_bias = pd.DataFrame(columns=[by,"Final PSA Bias", "Final PSA Bias p-value"])
    psa_recovery_bias = pd.DataFrame(columns=[by,"Recovery PSA Bias", "Recovery PSA Bias p-value"])
    psa_peak_bias = pd.DataFrame(columns=[by,"Peak PSA Bias", "Peak PSA Bias p-value"])
    # psa_peak_tap_bias = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Bias"])
    psa_avg_bias = pd.DataFrame(columns=[by,"Average PSA Bias", "Average PSA Bias p-value"])
    psa_sensitization_bias = pd.DataFrame(columns=[by,"Sensitization of PSA Bias", "Sensitization of PSA Bias p-value"])
    psa_habituation_bias = pd.DataFrame(columns=[by,"Habituation of PSA Bias", "Habituation of PSA Bias p-value"])
    psa_spontaneous_recovery_bias = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Bias", "Spontaneous Recovery of PSA Bias p-value"])
    psa_memory_retention_bias = pd.DataFrame(columns=[by,"Memory Retention of PSA Bias", "Memory Retention of PSA Bias p-value"])

    psa_initial_aspect = pd.DataFrame(columns=[by,"Initial PSA Aspect Ratio", "Initial PSA Aspect Ratio p-value"])
    psa_final_aspect = pd.DataFrame(columns=[by,"Final PSA Aspect Ratio", "Final PSA Aspect Ratio p-value"])
    psa_recovery_aspect = pd.DataFrame(columns=[by,"Recovery PSA Aspect Ratio", "Recovery PSA Aspect Ratio p-value"])
    psa_peak_aspect = pd.DataFrame(columns=[by,"Peak PSA Aspect Ratio", "Peak PSA Aspect Ratio p-value"])
    # psa_peak_tap_aspect = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Aspect Ratio"])
    psa_avg_aspect = pd.DataFrame(columns=[by,"Average PSA Aspect Ratio", "Average PSA Aspect Ratio p-value"])
    psa_sensitization_aspect = pd.DataFrame(columns=[by,"Sensitization of PSA Aspect Ratio", "Sensitization of PSA Aspect Ratio p-value"])
    psa_habituation_aspect = pd.DataFrame(columns=[by,"Habituation of PSA Aspect Ratio", "Habituation of PSA Aspect Ratio p-value"])
    psa_spontaneous_recovery_aspect = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Aspect Ratio", "Spontaneous Recovery of PSA Aspect Ratio p-value"])
    psa_memory_retention_aspect = pd.DataFrame(columns=[by,"Memory Retention of PSA Aspect Ratio", "Memory Retention of PSA Aspect Ratio p-value"])
    # psa_initial_to_peak_aspect = pd.DataFrame(columns=[by,"Initial_to_peak PSA Aspect Ratio"])
    # psa_peak_to_recovery_aspect = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Aspect Ratio"])

    psa_initial_kink = pd.DataFrame(columns=[by,"Initial PSA Kink", "Initial PSA Kink p-value"])
    psa_final_kink = pd.DataFrame(columns=[by,"Final PSA Kink", "Final PSA Kink p-value"])
    psa_recovery_kink = pd.DataFrame(columns=[by,"Recovery PSA Kink", "Recovery PSA Kink p-value"])
    psa_peak_kink = pd.DataFrame(columns=[by,"Peak PSA Kink", "Peak PSA Kink p-value"])
    # psa_peak_tap_kink = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Kink"])
    psa_avg_kink = pd.DataFrame(columns=[by,"Average PSA Kink", "Average PSA Kink p-value"])
    psa_sensitization_kink = pd.DataFrame(columns=[by,"Sensitization of PSA Kink", "Sensitization of PSA Kink p-value"])
    psa_habituation_kink = pd.DataFrame(columns=[by,"Habituation of PSA Kink", "Habituation of PSA Kink p-value"])
    psa_spontaneous_recovery_kink = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Kink", "Spontaneous Recovery of PSA Kink p-value"])
    psa_memory_retention_kink = pd.DataFrame(columns=[by,"Memory Retention of PSA Kink", "Memory Retention of PSA Kink p-value"])
    # psa_initial_to_peak_kink = pd.DataFrame(columns=[by,"Initial_to_peak PSA Kink"])
    # psa_peak_to_recovery_kink = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Kink"])

    psa_initial_curve = pd.DataFrame(columns=[by,"Initial PSA Curve", "Initial PSA Curve p-value"])
    psa_final_curve = pd.DataFrame(columns=[by,"Final PSA Curve", "Final PSA Curve p-value"])
    psa_recovery_curve = pd.DataFrame(columns=[by,"Recovery PSA Curve", "Recovery PSA Curve p-value"])
    psa_peak_curve = pd.DataFrame(columns=[by,"Peak PSA Curve", "Peak PSA Curve p-value"])
    # psa_peak_tap_curve = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Curve"])
    psa_avg_curve = pd.DataFrame(columns=[by,"Average PSA Curve", "Average PSA Curve p-value"])
    psa_sensitization_curve = pd.DataFrame(columns=[by,"Sensitization of PSA Curve", "Sensitization of PSA Curve p-value"])
    psa_habituation_curve = pd.DataFrame(columns=[by,"Habituation of PSA Curve", "Habituation of PSA Curve p-value"])
    psa_spontaneous_recovery_curve = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Curve", "Spontaneous Recovery of PSA Curve p-value"])
    psa_memory_retention_curve = pd.DataFrame(columns=[by,"Memory Retention of PSA Curve", "Memory Retention of PSA Curve p-value"])
    # psa_initial_to_peak_curve = pd.DataFrame(columns=[by,"Initial_to_peak PSA Curve"])
    # psa_peak_to_recovery_curve = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Curve"])

    psa_initial_crab = pd.DataFrame(columns=[by,"Initial PSA Crab", "Initial PSA Crab p-value"])
    psa_final_crab = pd.DataFrame(columns=[by,"Final PSA Crab", "Final PSA Crab p-value"])
    psa_recovery_crab = pd.DataFrame(columns=[by,"Recovery PSA Crab", "Recovery PSA Crab p-value"])
    psa_peak_crab = pd.DataFrame(columns=[by,"Peak PSA Crab", "Peak PSA Crab p-value"])
    # psa_peak_tap_crab = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Crab"])
    psa_avg_crab = pd.DataFrame(columns=[by,"Average PSA Crab", "Average PSA Crab p-value"])
    psa_sensitization_crab = pd.DataFrame(columns=[by,"Sensitization of PSA Crab", "Sensitization of PSA Crab p-value"])
    psa_habituation_crab = pd.DataFrame(columns=[by,"Habituation of PSA Crab", "Habituation of PSA Crab p-value"])
    psa_spontaneous_recovery_crab = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Crab", "Spontaneous Recovery of PSA Crab p-value"])
    psa_memory_retention_crab = pd.DataFrame(columns=[by,"Memory Retention of PSA Crab", "Memory Retention of PSA Crab p-value"])
    
    list_tap_Tstats = [
                    init_dura,
                    init_prob,
                    init_speed,
                    final_dura,
                    final_prob,
                    final_speed,
                    hab_dura,
                    hab_prob,
                    hab_speed,
                    recovery_dura,
                    recovery_prob,
                    recovery_speed,
                    memory_retention_dura,
                    memory_retention_prob,
                    memory_retention_speed,
                    psa_initial_speed,
                    psa_final_speed,
                    psa_recovery_speed,
                    psa_peak_speed,
                    # psa_peak_tap_speed,
                    psa_avg_speed,
                    psa_sensitization_speed,
                    psa_habituation_speed,
                    psa_spontaneous_recovery_speed,
                    psa_memory_retention_speed,
                    # psa_initial_to_peak_speed,
                    # psa_peak_to_recovery_speed,
                    psa_initial_bias,
                    psa_final_bias,
                    psa_recovery_bias,
                    psa_peak_bias,
                    # psa_peak_tap_bias,
                    psa_avg_bias,
                    psa_sensitization_bias,
                    psa_habituation_bias,
                    psa_spontaneous_recovery_bias,
                    psa_memory_retention_bias,
                    psa_initial_aspect,
                    psa_final_aspect,
                    psa_recovery_aspect,
                    psa_peak_aspect,
                    # psa_peak_tap_aspect,
                    psa_avg_aspect,
                    psa_sensitization_aspect,
                    psa_habituation_aspect,
                    psa_spontaneous_recovery_aspect,
                    psa_memory_retention_aspect,
                    # psa_initial_to_peak_aspect,
                    # psa_peak_to_recovery_aspect,
                    psa_initial_kink,
                    psa_final_kink,
                    psa_recovery_kink,
                    psa_peak_kink,
                    # psa_peak_tap_kink,
                    psa_avg_kink,
                    psa_sensitization_kink,
                    psa_habituation_kink,
                    psa_spontaneous_recovery_kink,
                    psa_memory_retention_kink,
                    # psa_initial_to_peak_kink,
                    # psa_peak_to_recovery_kink,
                    psa_initial_curve,
                    psa_final_curve,
                    psa_recovery_curve,
                    psa_peak_curve,
                    # psa_peak_tap_curve,
                    psa_avg_curve,
                    psa_sensitization_curve,
                    psa_habituation_curve,
                    psa_spontaneous_recovery_curve,
                    psa_memory_retention_curve,
                    # psa_initial_to_peak_curve,
                    # psa_peak_to_recovery_curve,
                    psa_initial_crab,
                    psa_final_crab,
                    psa_recovery_crab,
                    psa_peak_crab,
                    # psa_peak_tap_crab,
                    psa_avg_crab,
                    psa_sensitization_crab,
                    psa_habituation_crab,
                    psa_spontaneous_recovery_crab,
                    psa_memory_retention_crab]
    
    list_tap_metrics = [
                        "init_dura",
                        "init_prob",
                        "init_speed",

                        "final_dura",
                        "final_prob",
                        "final_speed",
                        
                        "habit_dura",
                        "habit_prob",
                        "habit_speed",
                        
                        "recovery_dura",
                        "recovery_prob",
                        "recovery_speed",

                        "memory_retention_dura",
                        "memory_retention_prob",
                        "memory_retention_speed",

                        "init_psa_speed",
                        "final_psa_speed",
                        "recov_psa_speed",
                        "max_psa_speed",
                        # "psa_peak_tap_speed",
                        "mean_psa_speed",
                        "sensitization_psa_speed",
                        "habit_psa_speed",
                        "recovery_psa_speed",
                        "memory_retention_psa_speed",
                        # "psa_initial_to_peak_speed",
                        # "psa_peak_to_recovery_speed",

                        "init_psa_bias",
                        "final_psa_bias",
                        "recov_psa_bias",
                        "max_psa_bias",
                        # "psa_peak_tap_bias",
                        "mean_psa_bias",
                        "sensitization_psa_bias",
                        "habit_psa_bias",
                        "recovery_psa_bias",
                        "memory_retention_psa_bias",

                        "init_psa_aspect_ratio",
                        "final_psa_aspect_ratio",
                        "recov_psa_aspect_ratio",
                        "max_psa_aspect_ratio",
                        # "psa_peak_tap_aspect_ratio",
                        "mean_psa_aspect_ratio",
                        "sensitization_psa_aspect_ratio",
                        "habit_psa_aspect_ratio",
                        "recovery_psa_aspect_ratio",
                        "memory_retention_psa_aspect_ratio",
                        # "psa_initial_to_peak_aspect_ratio",
                        # "psa_peak_to_recovery_aspect_ratio",

                        "init_psa_kink",
                        "final_psa_kink",
                        "recov_psa_kink",
                        "max_psa_kink",
                        # "psa_peak_tap_kink",
                        "mean_psa_kink",
                        "sensitization_psa_kink",
                        "habit_psa_kink",
                        "recovery_psa_kink",
                        "memory_retention_psa_kink",
                        # "psa_initial_to_peak_kink",
                        # "psa_peak_to_recovery_kink",

                        "init_psa_curve",
                        "final_psa_curve",
                        "recov_psa_curve",
                        "max_psa_curve",
                        # "psa_peak_tap_curve",
                        "mean_psa_curve",
                        "sensitization_psa_curve",
                        "habit_psa_curve",
                        "recovery_psa_curve",
                        "memory_retention_psa_curve",
                        # "psa_initial_to_peak_curve",
                        # "psa_peak_to_recovery_curve",

                        "init_psa_crab",
                        "final_psa_crab",
                        "recov_psa_crab",
                        "max_psa_crab",
                        # "psa_peak_tap_crab",
                        "mean_psa_crab",
                        "sensitization_psa_crab",
                        "habit_psa_crab",
                        "recovery_psa_crab",
                        "memory_retention_psa_crab"]
    
    return list_tap_Tstats, list_tap_metrics

In [ ]:
print(tap_data.columns.to_list())

In [ ]:
# def psa_metrics(by=["Gene", "dataset"]):
#     """
#     Create a list of empty dataframes and list of metric names for PSA summary analysis.

#     input:
#         by (list): what to group by ("Gene" or "dataset")

#     returns:
#         list_psa_Tstats: list of empty DataFrames for t-statistics
#         list_psa_metrics: list of metric names (short strings)
#     """


#     psa_initial_speed = pd.DataFrame(columns=[by,"Initial PSA Speed"])
#     psa_final_speed = pd.DataFrame(columns=[by,"Final PSA Speed"])
#     psa_recovery_speed = pd.DataFrame(columns=[by,"Recovery PSA Speed"])
#     psa_peak_speed = pd.DataFrame(columns=[by,"Peak PSA Speed"])
#     # psa_peak_tap_speed = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Speed"])
#     psa_avg_speed = pd.DataFrame(columns=[by,"Average PSA Speed"])
#     psa_sensitization_speed = pd.DataFrame(columns=[by,"Sensitization of PSA Speed"])
#     psa_habituation_speed = pd.DataFrame(columns=[by,"Habituation of PSA Speed"])
#     psa_spontaneous_recovery_speed = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Speed"])
#     psa_memory_retention_speed = pd.DataFrame(columns=[by,"Memory Retention of PSA Speed"])
#     # psa_initial_to_peak_speed = pd.DataFrame(columns=[by,"Initial_to_peak PSA Speed"])
#     # psa_peak_to_recovery_speed = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Speed"])

#     psa_initial_bias = pd.DataFrame(columns=[by,"Initial PSA Bias"])
#     psa_final_bias = pd.DataFrame(columns=[by,"Final PSA Bias"])
#     psa_recovery_bias = pd.DataFrame(columns=[by,"Recovery PSA Bias"])
#     psa_peak_bias = pd.DataFrame(columns=[by,"Peak PSA Bias"])
#     # psa_peak_tap_bias = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Bias"])
#     psa_avg_bias = pd.DataFrame(columns=[by,"Average PSA Bias"])
#     psa_sensitization_bias = pd.DataFrame(columns=[by,"Sensitization of PSA Bias"])
#     psa_habituation_bias = pd.DataFrame(columns=[by,"Habituation of PSA Bias"])
#     psa_spontaneous_recovery_bias = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Bias"])
#     psa_memory_retention_bias = pd.DataFrame(columns=[by,"Memory Retention of PSA Bias"])
#     # psa_initial_to_peak_bias = pd.DataFrame(columns=[by,"Initial_to_peak PSA Bias"])
#     # psa_peak_to_recovery_bias = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Bias"])

#     # psa_initial_ang_speed = pd.DataFrame(columns=[by,"Initial PSA Angular Speed"])
#     # psa_final_ang_speed = pd.DataFrame(columns=[by,"Final PSA Angular Speed"])
#     # psa_recovery_ang_speed = pd.DataFrame(columns=[by,"Recovery PSA Angular Speed"])
#     # psa_peak_ang_speed = pd.DataFrame(columns=[by,"Peak PSA Angular Speed"])
#     # # psa_peak_tap_ang_speed = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Angular Speed"])
#     # psa_avg_ang_speed = pd.DataFrame(columns=[by,"Average PSA Angular Speed"])
#     # psa_sensitization_ang_speed = pd.DataFrame(columns=[by,"Sensitization of PSA Angular Speed"])
#     # psa_habituation_ang_speed = pd.DataFrame(columns=[by,"Habituation of PSA Angular Speed"])
#     # psa_spontaneous_recovery_ang_speed = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Angular Speed"])
#     # psa_memory_retention_ang_speed = pd.DataFrame(columns=[by,"Memory Retention of PSA Angular Speed"])
#     # # psa_initial_to_peak_ang_speed = pd.DataFrame(columns=[by,"Initial_to_peak PSA Angular Speed"])
#     # # psa_peak_to_recovery_ang_speed = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Angular Speed"])

#     psa_initial_aspect = pd.DataFrame(columns=[by,"Initial PSA Aspect Ratio"])
#     psa_final_aspect = pd.DataFrame(columns=[by,"Final PSA Aspect Ratio"])
#     psa_recovery_aspect = pd.DataFrame(columns=[by,"Recovery PSA Aspect Ratio"])
#     psa_peak_aspect = pd.DataFrame(columns=[by,"Peak PSA Aspect Ratio"])
#     # psa_peak_tap_aspect = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Aspect Ratio"])
#     psa_avg_aspect = pd.DataFrame(columns=[by,"Average PSA Aspect Ratio"])
#     psa_sensitization_aspect = pd.DataFrame(columns=[by,"Sensitization of PSA Aspect Ratio"])
#     psa_habituation_aspect = pd.DataFrame(columns=[by,"Habituation of PSA Aspect Ratio"])
#     psa_spontaneous_recovery_aspect = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Aspect Ratio"])
#     psa_memory_retention_aspect = pd.DataFrame(columns=[by,"Memory Retention of PSA Aspect Ratio"])
#     # psa_initial_to_peak_aspect = pd.DataFrame(columns=[by,"Initial_to_peak PSA Aspect Ratio"])
#     # psa_peak_to_recovery_aspect = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Aspect Ratio"])

#     psa_initial_kink = pd.DataFrame(columns=[by,"Initial PSA Kink"])
#     psa_final_kink = pd.DataFrame(columns=[by,"Final PSA Kink"])
#     psa_recovery_kink = pd.DataFrame(columns=[by,"Recovery PSA Kink"])
#     psa_peak_kink = pd.DataFrame(columns=[by,"Peak PSA Kink"])
#     # psa_peak_tap_kink = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Kink"])
#     psa_avg_kink = pd.DataFrame(columns=[by,"Average PSA Kink"])
#     psa_sensitization_kink = pd.DataFrame(columns=[by,"Sensitization of PSA Kink"])
#     psa_habituation_kink = pd.DataFrame(columns=[by,"Habituation of PSA Kink"])
#     psa_spontaneous_recovery_kink = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Kink"])
#     psa_memory_retention_kink = pd.DataFrame(columns=[by,"Memory Retention of PSA Kink"])
#     # psa_initial_to_peak_kink = pd.DataFrame(columns=[by,"Initial_to_peak PSA Kink"])
#     # psa_peak_to_recovery_kink = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Kink"])

#     psa_initial_curve = pd.DataFrame(columns=[by,"Initial PSA Curve"])
#     psa_final_curve = pd.DataFrame(columns=[by,"Final PSA Curve"])
#     psa_recovery_curve = pd.DataFrame(columns=[by,"Recovery PSA Curve"])
#     psa_peak_curve = pd.DataFrame(columns=[by,"Peak PSA Curve"])
#     # psa_peak_tap_curve = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Curve"])
#     psa_avg_curve = pd.DataFrame(columns=[by,"Average PSA Curve"])
#     psa_sensitization_curve = pd.DataFrame(columns=[by,"Sensitization of PSA Curve"])
#     psa_habituation_curve = pd.DataFrame(columns=[by,"Habituation of PSA Curve"])
#     psa_spontaneous_recovery_curve = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Curve"])
#     psa_memory_retention_curve = pd.DataFrame(columns=[by,"Memory Retention of PSA Curve"])
#     # psa_initial_to_peak_curve = pd.DataFrame(columns=[by,"Initial_to_peak PSA Curve"])
#     # psa_peak_to_recovery_curve = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Curve"])

#     psa_initial_crab = pd.DataFrame(columns=[by,"Initial PSA Crab"])
#     psa_final_crab = pd.DataFrame(columns=[by,"Final PSA Crab"])
#     psa_recovery_crab = pd.DataFrame(columns=[by,"Recovery PSA Crab"])
#     psa_peak_crab = pd.DataFrame(columns=[by,"Peak PSA Crab"])
#     # psa_peak_tap_crab = pd.DataFrame(columns=[by,"Peak Tap Number of PSA Crab"])
#     psa_avg_crab = pd.DataFrame(columns=[by,"Average PSA Crab"])
#     psa_sensitization_crab = pd.DataFrame(columns=[by,"Sensitization of PSA Crab"])
#     psa_habituation_crab = pd.DataFrame(columns=[by,"Habituation of PSA Crab"])
#     psa_spontaneous_recovery_crab = pd.DataFrame(columns=[by,"Spontaneous Recovery of PSA Crab"])
#     psa_memory_retention_crab = pd.DataFrame(columns=[by,"Memory Retention of PSA Crab"])
#     # psa_initial_to_peak_crab = pd.DataFrame(columns=[by,"Initial_to_peak PSA Crab"])
#     # psa_peak_to_recovery_crab = pd.DataFrame(columns=[by,"Peak_to_recovery PSA Crab"])

#     list_psa_Tstats = [
#         # psa_initial_speed, psa_final_speed,psa_recovery_speed, psa_peak_speed, psa_peak_tap_speed, psa_avg_speed,
#         psa_initial_speed, psa_final_speed,psa_recovery_speed, psa_peak_speed, psa_avg_speed,
#         psa_sensitization_speed, psa_habituation_speed, psa_spontaneous_recovery_speed, psa_memory_retention_speed,

#         # psa_initial_bias, psa_final_bias, psa_recovery_bias, psa_peak_bias, psa_peak_tap_bias, psa_avg_bias,
#         psa_initial_bias, psa_final_bias, psa_recovery_bias, psa_peak_bias, psa_avg_bias,
#         psa_sensitization_bias, psa_habituation_bias, psa_spontaneous_recovery_bias, psa_memory_retention_bias,

#         # psa_initial_ang_speed, psa_final_ang_speed, psa_recovery_ang_speed, psa_peak_ang_speed, psa_avg_ang_speed,
#         # psa_sensitization_ang_speed, psa_habituation_ang_speed, psa_spontaneous_recovery_ang_speed, psa_memory_retention_ang_speed,

#         psa_initial_aspect, psa_final_aspect, psa_recovery_aspect, psa_peak_aspect, psa_avg_aspect,
#         psa_sensitization_aspect, psa_habituation_aspect, psa_spontaneous_recovery_aspect, psa_memory_retention_aspect,

#         psa_initial_kink, psa_final_kink, psa_recovery_kink, psa_peak_kink, psa_avg_kink,
#         psa_sensitization_kink, psa_habituation_kink, psa_spontaneous_recovery_kink, psa_memory_retention_kink,

#         psa_initial_curve, psa_final_curve, psa_recovery_curve, psa_peak_curve, psa_avg_curve,
#         psa_sensitization_curve, psa_habituation_curve, psa_spontaneous_recovery_curve, psa_memory_retention_curve,

#         psa_initial_crab, psa_final_crab, psa_recovery_crab, psa_peak_crab, psa_avg_crab,
#         psa_sensitization_crab, psa_habituation_crab, psa_spontaneous_recovery_crab, psa_memory_retention_crab
#     ]

#     list_psa_metrics = [
#     "Initial PSA Speed",
#     "Final PSA Speed",
#     "Recovery PSA Speed",
#     "Peak PSA Speed",
#     # "Peak Tap Number of PSA Speed",
#     "Average PSA Speed",
#     "Sensitization of PSA Speed",
#     "Habituation of PSA Speed",
#     "Spontaneous Recovery of PSA Speed",
#     "Memory Retention of PSA Speed",

#     "Initial PSA Bias",
#     "Final PSA Bias",
#     "Recovery PSA Bias",
#     "Peak PSA Bias",
#     # "Peak Tap Number of PSA Bias",
#     "Average PSA Bias",
#     "Sensitization of PSA Bias",
#     "Habituation of PSA Bias",
#     "Spontaneous Recovery of PSA Bias",
#     "Memory Retention of PSA Bias",

#     # "Initial PSA Angular Speed",
#     # "Final PSA Angular Speed",
#     # "Recovery PSA Angular Speed",
#     # "Peak PSA Angular Speed",
#     # # "Peak Tap Number of PSA Angular Speed",
#     # "Average PSA Angular Speed",
#     # "Sensitization of PSA Angular Speed",
#     # "Habituation of PSA Angular Speed",
#     # "Spontaneous Recovery of PSA Angular Speed",
#     # "Memory Retention of PSA Angular Speed",

#     "Initial PSA Aspect Ratio",
#     "Final PSA Aspect Ratio",
#     "Recovery PSA Aspect Ratio",
#     "Peak PSA Aspect Ratio",
#     # "Peak Tap Number of PSA Aspect Ratio",
#     "Average PSA Aspect Ratio",
#     "Sensitization of PSA Aspect Ratio",
#     "Habituation of PSA Aspect Ratio",
#     "Spontaneous Recovery of PSA Aspect Ratio",
#     "Memory Retention of PSA Aspect Ratio",


#     "Initial PSA Kink",
#     "Final PSA Kink",
#     "Recovery PSA Kink",
#     "Peak PSA Kink",
#     # "Peak Tap Number of PSA Kink",
#     "Average PSA Kink",
#     "Sensitization of PSA Kink",
#     "Habituation of PSA Kink",
#     "Spontaneous Recovery of PSA Kink",
#     "Memory Retention of PSA Kink",

#     "Initial PSA Curve",
#     "Final PSA Curve",
#     "Recovery PSA Curve",
#     "Peak PSA Curve",
#     # "Peak Tap Number of PSA Curve",
#     "Average PSA Curve",
#     "Sensitization of PSA Curve",
#     "Habituation of PSA Curve",
#     "Spontaneous Recovery of PSA Curve",
#     "Memory Retention of PSA Curve",

#     "Initial PSA Crab",
#     "Final PSA Crab",
#     "Recovery PSA Crab",
#     "Peak PSA Crab",
#     # "Peak Tap Number of PSA Crab",
#     "Average PSA Crab",
#     "Sensitization of PSA Crab",
#     "Habituation of PSA Crab",
#     "Spontaneous Recovery of PSA Crab",
#     "Memory Retention of PSA Crab"
# ]
    
#     return list_psa_Tstats, list_psa_metrics


In [ ]:
def TTest(Type, DF_ref, output, by=["Gene", "dataset"]):
    """
    Perform two sample t-test for each unique Gene/dataset column in the Df_ref
    input: 
        - a:column name of values 
        - DF_ref:reference dataframe
        - output: output df to store results in 
        - by: what to group by "Gene" or "dataset"
        
    """
    for a in DF_ref[by].unique():
        Tstat_a = ttest_ind(DF_ref[DF_ref.dataset == a][Type], DF_ref[DF_ref.Allele.isin(["XJ1","N2"])][Type],equal_var=False)[0]
        # print("This is Tstat_a:", Tstat_a)
        Tstat_a_pval = ttest_ind(DF_ref[DF_ref.dataset == a][Type], DF_ref[DF_ref.Allele.isin(["XJ1","N2"])][Type],equal_var=False)[1]
        # print("This is Tstat_a_pval:", Tstat_a_pval)
        Tstat_g = ttest_ind(DF_ref[DF_ref.Gene == a][Type], DF_ref[DF_ref.Gene == "N2"][Type],equal_var=False)[0]
        # print("This is Tstat_g:", Tstat_g)
        Tstat_g_pval = ttest_ind(DF_ref[DF_ref.Gene == a][Type], DF_ref[DF_ref.Gene == "N2"][Type],equal_var=False)[1]
        # print("This is Tstat_g_pval:", Tstat_g_pval)
        Tstat = Tstat_g if by=="Gene" else Tstat_a
        Tstat_pval = Tstat_g_pval if by=="Gene" else Tstat_a_pval
        row = [a, Tstat, Tstat_pval]
        # print(row)
        output.loc[len(output)] = row
    # print(output)

def do_TTest(by=["Gene", "dataset"], baseline=["true", "false", "psa"]):
    """
    Perform TTest function for each unique Gene/dataset column in baseline_output/tap_data
    
    input: 
        - by: what to group by "Gene" or "dataset"
        - baseline: whether or not to use baseline data

    returns: sorted T-statistics dataframe
    """

    if baseline=="true":
        list_Tstats, list_metrics = baseline_metrics(by)
        data = baseline_output
    elif baseline=="false":
        list_Tstats,list_metrics = tap_metrics(by)
        data = tap_data.dropna()
    else:
        pass
        # list_Tstats,list_metrics = psa_metrics(by)
        # data = psa_data
    for x in data[by].unique():
        if Screen=="Neuron_Genes_Screen":
            condition = x in (["N2"] if by == "Gene" else ["N2_XJ1", "N2_N2"])
        else:
            condition = (x =="N2")
        if condition:
            pass
        else:
            output_gene=data[data[by]==x]
            gene_data=data[data['Date'].isin(output_gene['Date'].unique())]
            if Screen=="Neuron_Genes_Screen":
                gene_data_final = gene_data[gene_data[by].isin(['N2', x])] if by=="Gene" else gene_data[gene_data[by].isin(['N2_N2','N2_XJ1', x])]
            else:
                gene_data_final = gene_data[gene_data[by].isin(['N2', x])]

            for a,b in zip(list_metrics, list_Tstats):
                TTest(a, gene_data_final, b, by) # calls t test function
    
    PD_Tstats=pd.DataFrame()
    for a in list_Tstats:
        b=a.groupby([by], as_index=False).mean()
        if b.columns.values[1] == list_Tstats[0].columns.values[1]:
            PD_Tstats=b
        else:
            PD_Tstats=PD_Tstats.join(b.iloc[:,1:3])
            
    PD_Tstats=PD_Tstats.set_index(by)
    
    return PD_Tstats
            

### T-stat on Baseline data:

### 3.3.1 Allele-level T-stat analysis of baseline data

In [ ]:
def pair_pvals(df):
    """
    Extract p-value columns from a DataFrame and apply FDR correction.

    input:
        - df: DataFrame containing T-statistics and p-values in alternating columns
    returns:
        - fdr_corrected: DataFrame with FDR-corrected p-values
    """

    # Extract p-value columns (every second column starting from index 1)
    Tstats_pvals = df.iloc[:, 1::2].copy()
    df_final = df.iloc[:, ::2].copy()

    # Rename columns to remove '_x' suffix if present and clarify they are p-values
    Tstats_pvals.columns = [col if 'p-value' in col else col + ' p-value' for col in Tstats_pvals.columns]
    # Tstats_pvals.head()

    # Apply FDR correction to each row of Tstats_pvals
    fdr_corrected = Tstats_pvals.apply(
        lambda row: pd.Series(fdr(row.dropna(), alpha=0.1)[1], index=row.dropna().index),
        axis=1
    )

    # Remove " p-value" from each column label in fdr_corrected
    fdr_corrected.columns = [col.replace(" p-value", "") for col in fdr_corrected.columns]

    Tstats_with_fdr = pd.DataFrame(
    index=df_final.index,
    columns=df_final.columns
    )

    for col in df_final.columns:
        Tstats_with_fdr[col] = list(
            zip(
                df_final[col],
                fdr_corrected[col] if col in fdr_corrected.columns else [None] * len(df_final)
            )
        )
        Tstats_with_fdr[col] = list(
            zip(
                df_final[col],
                fdr_corrected[col] if col in fdr_corrected.columns else [None] * len(df_final)
            )
        )

    # Tstats_with_fdr.head()

    return Tstats_with_fdr
# fdr_corrected.head()

In [ ]:
warnings.filterwarnings('ignore')

PD_baseline_Tstats_allele = do_TTest("dataset", baseline="true") # get sorted T-statistics DataFrame 
PD_baseline_Tstats_allele_final = pair_pvals(PD_baseline_Tstats_allele)
# PD_baseline_Tstats_allele_sorted=PD_baseline_Tstats_allele.sort_index()

PD_baseline_Tstats_allele_final.head()

### 3.3.2 Gene-level T-stat analysis of baseline data

In [ ]:
warnings.filterwarnings('ignore')

PD_baseline_Tstats=do_TTest("Gene", baseline="true") # get sorted T-statistics DataFrame 
PD_baseline_Tstats_final = pair_pvals(PD_baseline_Tstats)

# PD_baseline_Tstats_sorted=PD_baseline_Tstats.sort_index()

PD_baseline_Tstats_final.head()

In [ ]:
# PD_baseline_Tstats
PD_baseline_Tstats_final.columns.to_list()
len(PD_baseline_Tstats_final.columns.to_list())

### T-stat analysis for tap-response data:

### 3.3.3 Allele level T-stat analysis of tap response data

In [ ]:
tap_data

In [ ]:
warnings.filterwarnings('ignore')

PD_habituation_Tstats_allele = do_TTest("dataset", baseline="false") # get sorted T-statistics DataFrame 
PD_habituation_Tstats_allele_final = pair_pvals(PD_habituation_Tstats_allele)
# PD_habituation_Tstats_allele_sorted=PD_habituation_Tstats_allele.sort_index()

PD_habituation_Tstats_allele_final.head()

In [ ]:
PD_habituation_Tstats_allele_final.columns.to_list()
len(PD_habituation_Tstats_allele_final.columns.to_list())

### 3.3.4 Gene-level T-stat analysis of Tap response data

In [ ]:
warnings.filterwarnings('ignore')

PD_habituation_Tstats = do_TTest("Gene", baseline="false") # get sorted T-statistics DataFrame 
PD_habituation_Tstats_final = pair_pvals(PD_habituation_Tstats)


PD_habituation_Tstats_final.head()

In [ ]:
# # Extract p-value columns (every second column starting from index 1)
# PD_habituation_Tstats_pvals = PD_habituation_Tstats.iloc[:, 1::2].copy()
# PD_habituation_Tstats_final = PD_habituation_Tstats.iloc[:, ::2].copy()

# # Rename columns to remove '_x' suffix if present and clarify they are p-values
# PD_habituation_Tstats_pvals.columns = [col if 'p-value' in col else col + ' p-value' for col in PD_habituation_Tstats_pvals.columns]

# PD_habituation_Tstats_pvals.head()

In [ ]:
# # Apply FDR correction to each row of PD_habituation_Tstats_pvals
# fdr_corrected = PD_habituation_Tstats_pvals.apply(
#     lambda row: pd.Series(fdr(row.dropna(), alpha=0.1)[1], index=row.dropna().index),
#     axis=1
# )

# # Remove " p-value" from each column label in fdr_corrected
# fdr_corrected.columns = [col.replace(" p-value", "") for col in fdr_corrected.columns]

# fdr_corrected.head()

In [ ]:
# # Combine T-statistics with FDR-corrected p-values into tuples
# PD_habituation_Tstats_with_fdr = pd.DataFrame(
#     index=PD_habituation_Tstats_final.index,
#     columns=PD_habituation_Tstats_final.columns
# )

# for col in PD_habituation_Tstats_final.columns:
#     PD_habituation_Tstats_with_fdr[col] = list(
#         zip(
#             PD_habituation_Tstats_final[col],
#             fdr_corrected[col] if col in fdr_corrected.columns else [None] * len(PD_habituation_Tstats_final)
#         )
#     )

# PD_habituation_Tstats_with_fdr.head()

### T-stat analysis for psa data:

### 3.3.5 Allele level T-stat analysis of PSA data * defunct *

In [ ]:
# warnings.filterwarnings('ignore')

# psa_tstats_allele = do_TTest("dataset", baseline="psa") # get sorted T-statistics DataFrame 

# psa_tstats_allele.head()

### 3.3.6 Gene-level T-stat analysis of PSA data

In [ ]:
# warnings.filterwarnings('ignore')

# psa_tstats = do_TTest("Gene", baseline="psa") # get sorted T-statistics DataFrame 

# psa_tstats.head()

# 4. Merging t-stat data into one dataset

In [ ]:
def pop_cols(combined):
    """
    Reorders columns in the combined dataframe. 
    (pops specific columns["Area", "Midline", "Morphwidth", "Angular Speed"] and
    reinserts at different positions)

    input:
        combined: dataframe with columns to be reordered

    returns: 
        NA    
        
    """
    first_col=combined.pop("Area")
    combined.insert(0,"Area",first_col)

    first_col=combined.pop("Midline")
    combined.insert(0,"Midline",first_col)

    first_col=combined.pop("Morphwidth")
    combined.insert(0,"Morphwidth",first_col)

    first_col=combined.pop("Angular Speed")
    combined.insert(5,"Angular Speed",first_col)

def pop_last(combined):
    """
    Reorders the last three columns of the combined dataframe.
    input:
        combined: dataframe with columns to be reordered

    """
    last_col=combined.pop("Spontaneous Recovery of Response Duration")
    combined.insert(25,"Spontaneous Recovery of Response Duration",last_col)

    last_col=combined.pop("Spontaneous Recovery of Response Probability")
    combined.insert(25,"Spontaneous Recovery of Response Probability",last_col)

    last_col=combined.pop("Spontaneous Recovery of Response Speed")
    combined.insert(25,"Spontaneous Recovery of Response Speed",last_col)

    last_col=combined.pop("Memory Retention of Response Duration")
    combined.insert(25,"Memory Retention of Response Duration",last_col)

    last_col=combined.pop("Memory Retention of Response Probability")
    combined.insert(25,"Memory Retention of Response Probability",last_col)

    last_col=combined.pop("Memory Retention of Response Speed")
    combined.insert(25,"Memory Retention of Response Speed",last_col)

def rename_columns(df):
    '''
    Renames columns in the input dataframe
    input:
        combined: dataframe with columns to be renamed   
    returns:
        input dataframe with renamed columns 
    '''
    renames = {
        "Habituation of Duration": "Habituation of Response Duration",
        "Habituation of Probability": "Habituation of Response Probability",
        "Habituation of Speed": "Habituation of Response Speed",
        "Initial Duration": "Initial Response Duration",
        "Initial Probability": "Initial Response Probability",
        "Initial Speed": "Initial Response Speed",
        "Final Duration": "Final Response Duration",
        "Final Probability": "Final Response Probability",
        "Final Speed": "Final Response Speed",
        "Recovery Duration": "Spontaneous Recovery of Response Duration",
        "Recovery Probability": "Spontaneous Recovery of Response Probability",
        "Recovery Speed": "Spontaneous Recovery of Response Speed",
        "Memory Retention Duration": "Memory Retention of Response Duration",
        "Memory Retention Probability": "Memory Retention of Response Probability",
        "Memory Retention Speed": "Memory Retention of Response Speed"
    }
    return df.rename(columns=renames)

def merge_Tstats(baseline, habituation, by=["Gene", "dataset"], Screen=Screen, psa=False):
    """
    merge baseline and tap response dataframes based on the Gene/dataset
    normalize the merged dataframe and then return it with melted version

    input:
        - baseline: baseline dataframe to merge
        - habituation: habituation dataframe to merge
        - by: what to group by "Gene" or "dataset"
    """

    #merge baseline and habituation data
    combined_Tstats = pd.merge(baseline, habituation, on=by, how='left')
    combined_Tstats = combined_Tstats.sort_index() # sort by index

    # ------------ NORMALISATION STEPS MOVED TO DASHBOARD -------------------
    # # normalise combined dataframe by subtracting mean and div by sd
    # combined_Tstats_normalized = (combined_Tstats-combined_Tstats.mean())/combined_Tstats.std()

    # if by=="dataset" and Screen=="Neuron_Genes_Screen":
    #     combined_Tstats_normalized_2 = combined_Tstats-combined_Tstats[combined_Tstats.index=="N2_XJ1"].squeeze()
    # else :
    #     combined_Tstats_normalized_2 = combined_Tstats-combined_Tstats[combined_Tstats.index=="N2"].squeeze()  

    # pop_cols(combined_Tstats) # reorder columns

    # Skip this step if data = psa
    if not psa:
        #rename columns of combined and normalized df
        combined_Tstats = rename_columns(combined_Tstats)
        # combined_Tstats_normalized_2=rename_columns(combined_Tstats_normalized_2)
        # pop_cols(combined_Tstats) # reorder columns
        # pop_last(combined_Tstats) # reorder columns

    # -------------- PIVOTING STEPS MOVED TO DASHBOARD ---------------------
    # # Melt the combined dataframe
    # combined_Tstats_melted=combined_Tstats.reset_index()
    # combined_Tstats_melted=pd.melt(combined_Tstats_melted, id_vars=[by],
    #                             var_name='Metric',
    #                             value_name='T_score')
    
    # # Sort the melted dataframe by T_score
    # combined_Tstats_melted_sorted=combined_Tstats_melted.sort_values(by=['T_score'])

    # # Melt the normalized dataframe
    # combined_Tstats_normalized_melted=combined_Tstats.reset_index()
    # combined_Tstats_normalized_melted=pd.melt(combined_Tstats_normalized_melted, id_vars=[by],
    #                                                var_name='Metric',
    #                                                value_name='T_score')

    # add Screen column to df and its melted version
    combined_Tstats['Screen']=Screen
    combined_Tstats = reorder_screencolumn(combined_Tstats)
    # combined_Tstats_normalized_melted['Screen']=Screen

    return combined_Tstats#, combined_Tstats_normalized_melted

def reorder_screencolumn(df):
    """
    Reorders the 'Screen' column to be the second column in the dataframe.

    input:
        df: DataFrame with a 'Screen' column

    returns:
        DataFrame with 'Screen' as the second column
    """
    cols = df.columns.tolist()
    if 'Screen' in cols:
        cols.insert(0, cols.pop(cols.index('Screen')))
        df = df[cols]
    return df

## 4.1 Gene-level

- Pass Tap and baseline through merge_Tstats() as df1
- Pass PSA and baseline through merge_Tstats()as df2
- pd.merge df1 and df2 using all columns of baseline

In [ ]:
# Baseline + Tap
combined_Tstats = merge_Tstats(PD_baseline_Tstats_final, PD_habituation_Tstats_final, "Gene")

In [ ]:
# # Baseline + PSA 
# combined_Tstats_psa = merge_Tstats(
#     PD_baseline_Tstats, psa_tstats, by="Gene", psa=True
# )

In [ ]:
# Baseline + Tap + PSA
# final_tstat = pd.merge(combined_Tstats.reset_index(), combined_Tstats_psa.reset_index(), on = PD_baseline_Tstats.columns.to_list().append(['Gene','Screen']), how = 'inner')
final_tstat = combined_Tstats.copy()
# final_tstat = reorder_screencolumn(final_tstat)
final_tstat.head()

In [ ]:
final_tstat

In [ ]:
# # Baseline + Tap + PSA melted
# final_tstat_melted = pd.concat([combined_Tstats_normalized_melted, combined_Tstats_psa_melted]).drop_duplicates()

# final_tstat_melted.head()

## 4.2 Allele level 


- Pass Tap and baseline through merge_Tstats() as df3
- Pass PSA and baseline through merge_Tstats()as df4
- pd.merge df3 and df4 using all columns of basline

In [ ]:
# Baseline + Tap
combined_Tstats_allele = merge_Tstats(PD_baseline_Tstats_allele_final,PD_habituation_Tstats_allele_final, "dataset")

In [ ]:
# # Baseline + PSA 
# combined_Tstats_psa_allele = merge_Tstats(
#     PD_baseline_Tstats_allele, psa_tstats_allele, by="dataset", psa=True
# )

In [ ]:
# Baseline + Tap + PSA
# final_tstat_allele = pd.merge(combined_Tstats_allele.reset_index(), combined_Tstats_psa_allele.reset_index(), on = PD_baseline_Tstats_allele.columns.to_list().append(['dataset','Screen']), how = 'outer')
final_tstat_allele = combined_Tstats_allele.copy()
# final_tstat_allele = reorder_screencolumn(final_tstat_allele)
final_tstat_allele.head()

In [ ]:
final_tstat.shape

In [ ]:
# combined_MSD = reorder_screencolumn(combined_MSD)
# allele_combined_MSD = reorder_screencolumn(allele_combined_MSD)

In [ ]:
# # Baseline + Tap + PSA melted
# final_tstat_melted_allele = pd.concat([combined_Tstats_normalized_melted_allele, combined_Tstats_psa_melted_allele]).drop_duplicates()

# final_tstat_melted_allele.head()

# 5. Save data to database (sqlite3)

#### A janky way to add data and update the sql 

1. Read table to pd.DataFrame
2. Add new data to pd.DataFrame
3. Replace old table with newly updated pd.DataFrame

# Primary Keys For Each SQL Table:

####  -- Gene_Allele_WormBaseID:
WBGene, WBAllele
#### -- alleleMSD:
dataset, Screen
#### -- gene_MSD:
Gene, Screen
#### -- allele_profile_data:
dataset, Metric, Screen
#### -- gene_profile_data:
Gene, Metric, Screen
#### -- tap_baseline_data:
Time, Plate_id, Date, Screen, dataset, Gene, Allele
#### -- tap_response_data:
Date, Plate_id, Screen, taps, dataset, Gene, Allele
#### -- tstat_allele_data:
dataset, Screen
#### -- tstat_gene_data:
Gene, Screen
#### -- psa_summarized_data:
Plate_id, Date ,Screen ,dataset ,Gene ,Allele

In [ ]:
# print(tap_output.head(5))
# print(baseline_output.head(5))

tap_output.Screen = Screen
tap_psa_output.Screen = Screen
baseline_output.Screen = Screen

# print(tap_output.head(5))
# print(baseline_output.head(5))

In [ ]:
# final_tstat_allele[final_tstat_allele.isna().any(axis=1)]
final_tstat_allele[final_tstat_allele["Morphwidth"].isna()]

In [ ]:
# final_tstat_allele[final_tstat_allele['dataset'] == "unknown_CZ11000"]

## Reorder columns before uploading

In [ ]:
baseline_output = pd.concat([baseline_output[['Screen', 'dataset', 'Gene', 'Allele', 'Plate_id', 'Date']], 
                             baseline_output.drop(columns=['Screen', 'dataset', 'Gene', 'Allele', 'Plate_id', 'Date'])], axis=1)

tap_psa_output = pd.concat([tap_psa_output[['Screen', 'dataset', 'Gene', 'Allele', 'Plate_id', 'Date', 'taps']], 
                            tap_psa_output.drop(columns=['Screen', 'dataset', 'Gene', 'Allele', 'Plate_id', 'Date', 'taps'])],axis=1)

final_tstat = pd.concat([final_tstat[['Screen', 'Gene']], 
                         final_tstat.drop(columns = ['Screen', 'Gene'])], axis=1)

final_tstat_allele = pd.concat([final_tstat_allele[['Screen', 'dataset']], 
                                final_tstat_allele.drop(columns = ['Screen', 'dataset'])], axis=1)

combined_MSD = pd.concat([combined_MSD[['Screen', 'Gene']], 
                         combined_MSD.drop(columns = ['Screen', 'Gene'])], axis=1)

allele_combined_MSD = pd.concat([allele_combined_MSD[['Screen','dataset']], 
                         allele_combined_MSD.drop(columns = ['Screen','dataset'])], axis=1)

psa_data = pd.concat([psa_data[['Screen', 'dataset', 'Gene', 'Allele', 'Plate_id', 'Date']], 
                      psa_data.drop(columns=['Screen', 'dataset', 'Gene', 'Allele', 'Plate_id', 'Date'])], axis=1)


## Save to database

In [ ]:

### This code will connect to PostgreSQL database and write non-duplicate data into the database tables.

# Loads database config values from database.ini file and validates that user and password are set.
config = load_config()
if (config['user'] == "" or config['password'] == ""):
    print("Please set your user and password in the database.ini file.")
    sys.exit(1)
    
# Creates a connection pool to PostgreSQL database using SQLAlchemy.
engine = create_engine(f"postgresql+psycopg://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}")

# Function to insert data into PostgreSQL table, skipping duplicates based on primary keys.
def postgres_skip_on_duplicate(pd_table, conn, keys, data_iter):
    data = [dict(zip(keys,row)) for row in data_iter]
    conn.execute(insert(pd_table.table).on_conflict_do_nothing(), data)

# --------- Write the dataframes to PostgreSQL tables -----------

# Complete tap response data
print("working on tap_psa_output:") 
tap_psa_output.to_sql('tap_response_data', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)
# tap_psa_output.to_sql('tap_response_data', engine, if_exists='replace', index=False, method=None)

# Complete baseline data  >NO
print("working on tap_baseline_data:") 
baseline_output.to_sql('tap_baseline_data', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)
# baseline_output.to_sql('tap_baseline_data', engine, if_exists='replace', index=False, method=None)

# Baseline + Tap + PSA combined tstat data by Gene
print("working on tstat_gene_data")
final_tstat.dropna(thresh=10).reset_index().to_sql('tstat_gene_data', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)
# final_tstat.reset_index().to_sql('tstat_gene_data', engine, if_exists='replace', index=False, method=None)

# Baseline + Tap + PSA combined tstat data by Allele
print("working on tstat_allele_data")
final_tstat_allele.dropna(thresh=10).reset_index().to_sql('tstat_allele_data', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)
# final_tstat_allele.reset_index().to_sql('tstat_allele_data', engine, if_exists='replace', index=False, method=None)

# MSD Baseline + Tap + PSA by Gene
print("working on gene_MSD")
combined_MSD.to_sql('gene_MSD', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)
# combined_MSD.to_sql('gene_MSD', engine, if_exists='replace', index=False, method=None)

# MSD Baseline + Tap + PSA by Allele
print("working on allele_MSD")
allele_combined_MSD.to_sql('allele_MSD', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)
# allele_combined_MSD.to_sql('allele_MSD', engine, if_exists='replace', index=False, method=None)

# Summarised PSA data (speed, kink, curve, etc.)
print("working on psa_data:") 
psa_data.to_sql('psa_summarised_data', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)
# psa_data.to_sql('psa_summarised_data', engine, if_exists='replace', index=False, method=None)

# # Melted Baseline + Tap + PSA combined tstat data by Gene
# print("working on gene_profile_data")
# final_tstat_melted.to_sql('gene_profile_data', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)

# # Melted Baseline + Tap + PSA combined tstat data by Allele
# print("working on allele_profile_data")
# final_tstat_melted_allele.to_sql('allele_profile_data', engine, if_exists='append', index=False, method=postgres_skip_on_duplicate)


print("---------- DONE ----------")

### Use the below cell if you want output to local .csv file

In [ ]:
# tap_psa_output.to_csv('/Users/Joseph/Desktop/PDScreen_tap_psa_output.csv', index=False)
# final_tstat.to_csv('/Users/Joseph/Desktop/PDScreen_final_tstat.csv', index=False)
# final_tstat_allele.to_csv('/Users/Joseph/Desktop/PDScreen_final_tstat_allele.csv', index=False)
# combined_MSD.to_csv('/Users/Joseph/Desktop/PDScreen_combined_MSD.csv', index=False)
# allele_combined_MSD.to_csv('/Users/Joseph/Desktop/PDScreen_combined_MSD_allele.csv', index=False)
# psa_data.to_csv('/Users/Joseph/Desktop/PDScreen_psa_data.csv', index=False)

### Use the below cell to just replace/update one table:

In [ ]:
# Loads database config values from database.ini file and validates that user and password are set.
config = load_config()
if (config['user'] == "" or config['password'] == ""):
    print("Please set your user and password in the database.ini file.")
    sys.exit(1)
    
# Creates a connection pool to PostgreSQL database using SQLAlchemy.
engine = create_engine(f"postgresql+psycopg://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}")

# Function to insert data into PostgreSQL table, skipping duplicates based on primary keys.
def postgres_skip_on_duplicate(pd_table, conn, keys, data_iter):
    data = [dict(zip(keys,row)) for row in data_iter]
    conn.execute(insert(pd_table.table).on_conflict_do_nothing(), data)


# Complete tap response data
print("working on tap_output:") 
tap_psa_output.to_sql('tap_response_data', engine, if_exists='replace', index=False, method=None)
print("Done")

In [ ]:
# # USE THIS CELL TO UPDATE ALL THE NEED TALBES (Also have baseline_output on the second line)

# conn=sqlite3.connect('/Users/lavanya/Desktop/Lavanya_Test/data_updated2.db')

# tap_output.to_sql('tap_response_data', conn, if_exists='append', index=False)

# baseline_output.to_sql('tap_baseline_data', conn, if_exists='append', index=False)

# combined_Tstats_normalize_2.reset_index().to_sql('tstat_gene_data', conn, if_exists='append', index=False)

# combined_Tstats_normalize_allele_2.reset_index().to_sql('tstat_allele_data', conn, if_exists='append', index=False)

# combined_Tstats_normalized_melted.to_sql('gene_profile_data', conn, if_exists='append', index=False)

# combined_Tstats_normalized_melted_allele.to_sql('allele_profile_data', conn, if_exists='append', index=False)

# combined_MSD.to_sql('gene_MSD', conn, if_exists='append', index=False)

# allele_combined_MSD.to_sql('allele_MSD', conn, if_exists='append', index=False)

# # combined_Tstats_melted_sorted.to_sql('allele_phenotype_data', conn, if_exists='replace', index=False)

# print(conn.total_changes)

# conn.close()


# # Want to test edge cases of pd.to_sql functionality#############